# 3.1 — Validação com dados reais de motor (CbM-Datasets, Analog Devices)

Os notebooks `2.x` dependem da sua coleta física — ainda não rodam de ponta a ponta sem ela.
Este notebook usa um **dataset público de vibração de motor real** para validar a MESMA lógica
de janelamento/features/split, com dados que ninguém aqui gerou: bancada de falhas
**SpectraQuest Machinery Fault Simulator**, acelerômetro **Analog Devices ADXL356**, publicado
em [`analogdevicesinc/CbM-Datasets`](https://github.com/analogdevicesinc/CbM-Datasets).

> **Trilha `3.x`, não `2.x`:** isto não é "Forma 1" nem "Forma 2" do resto do `app17-IMU` — é
> uma validação independente, com um motor de bancada de verdade, não com o MPU6050/6500 da
> aula. Não depende de coleta nenhuma: baixa os CSVs direto do GitHub.

## Avisos obrigatórios antes de usar estes dados

- **As unidades são VOLTS, não m/s².** Os valores ficam em torno de 0,89–0,91 nos três eixos —
  é o bias ratiométrico de saída do ADXL356 (≈ Vs/2 ≈ 0,9 V), não gravidade. Isso não afeta a
  classificação (o `StandardScaler` normaliza), mas não é m/s² e não deve ser chamado assim.
- **O repositório não declara licença.** Por isso baixamos em runtime direto do GitHub, com
  atribuição à Analog Devices, e **não versionamos** os CSVs neste repositório.

In [ ]:
!pip install -q pandas numpy scipy scikit-learn matplotlib

## 1) Baixar o subconjunto

O dataset completo tem 130 arquivos (13 configurações de falha × 5 RPMs × 2 cargas, 229 MB).
Para uma sessão de Colab leve, usamos só `00lb` (sem carga extra) e **4 configurações** — as
mesmas já usadas como referência mais adiante: saudável, desbalanceamento muito pesado, pista
interna de rolamento pesada, e eixo empenado. Para cada uma, baixamos as **5 velocidades**
(600 a 3000 RPM): é isso que nos dá várias "rodadas" por classe — o dataset não tem repetições
na mesma condição, então tratamos a variação de RPM como a diversidade entre coletas (o mesmo
papel que a coluna `rodada` cumpre nos notebooks `2.x`).

20 arquivos, ~36 MB. Convenção de nome: `RPM_rolamento_eixo_desbalanceamento_alinhamento_carga.Wfm.csv`
(ex.: `1800_GoB_GS_BaLo_WA_00lb.Wfm.csv` = 1800 RPM, rolamento bom, eixo bom, balanceado, bem
alinhado, sem carga extra).

In [ ]:
import os, urllib.request
import pandas as pd, numpy as np

BASE_URL = ("https://raw.githubusercontent.com/analogdevicesinc/CbM-Datasets/main/"
           "SampleMotorDataLimerick/SpectraQuest_Rig_Data_Voyager_3/Data_ADXL356C")
PASTA_CACHE = "cbm_cache"
os.makedirs(PASTA_CACHE, exist_ok=True)

# codigo da convencao de nome -> label legivel (as 4 configuracoes usadas neste notebook)
CLASSES = {
    "saudavel":          "GoB_GS_BaLo_WA",
    "desbalanceamento":  "GoB_GS_VHIL_WA",   # desbalanceamento muito pesado
    "rolamento_interno": "HIR_GS_BaLo_WA",   # pista interna, falha pesada
    "eixo_empenado":     "GoB_BS_BaLo_WA",
}
RPMS  = ["0600", "1200", "1800", "2400", "3000"]
CARGA = "00lb"

def baixar(nome_arquivo):
    destino = os.path.join(PASTA_CACHE, nome_arquivo)
    if not os.path.exists(destino):
        urllib.request.urlretrieve(f"{BASE_URL}/{nome_arquivo}", destino)
    return destino

arquivos = []  # (label, rpm, caminho_local)
for label, codigo in CLASSES.items():
    for rpm in RPMS:
        nome = f"{rpm}_{codigo}_{CARGA}.Wfm.csv"
        arquivos.append((label, rpm, baixar(nome)))

print(f"{len(arquivos)} arquivos baixados/cacheados em {PASTA_CACHE}/")

## 2) Formato do CSV e sanity check

Cada arquivo: separador `;`, **sem cabeçalho**, coluna 0 = tempo (s), colunas 1–3 = X/Y/Z.
A primeira linha do arquivo carrega 3 campos extras (ignorados via `usecols`). `fs = 20 kHz`,
2 segundos por arquivo, 40.000 amostras.

Antes de seguir, reproduzimos as estatísticas (1800 RPM, eixo X) já medidas na especificação
deste notebook — se baterem, a leitura está correta:

| condição | média x | std x | curtose x |
|---|---|---|---|
| saudável | 0,8911 | 0,00964 | 0,27 |
| desbalanceamento muito pesado | 0,8909 | 0,01659 | 0,07 |
| pista interna pesada | 0,8883 | 0,05292 | **18,69** |
| eixo empenado | 0,8913 | 0,01020 | 0,26 |

In [ ]:
def carregar_wfm(caminho):
    df = pd.read_csv(caminho, sep=";", header=None, usecols=[0, 1, 2, 3],
                     names=["t", "x", "y", "z"])
    return df.astype(float)

print("Sanity check (1800 RPM, eixo X) — compare com a tabela acima:")
for label, rpm, caminho in arquivos:
    if rpm != "1800":
        continue
    df = carregar_wfm(caminho)
    print(f"  {label:20s} media={df['x'].mean():.4f}  std={df['x'].std():.5f}  "
         f"curtose={df['x'].kurtosis():.2f}")

## 3) Janelamento (20 kHz) e features

Janelas de **4096 amostras** (~0,2 s), sem sobreposição: ~9 janelas por arquivo, ~180 no total.
**Janelas do mesmo arquivo não são independentes** — por isso mantemos a coluna `rpm` como
identificador de grupo, do mesmo jeito que `rodada` nos notebooks `2.x`, para usar depois num
`LeaveOneGroupOut` sem vazamento.

Mesmas features dos notebooks `2.x` (magnitude `mag = sqrt(x²+y²+z²)`, componente AC
`mag - mean(mag)`), mais `mean_x/y/z` — aqui de propósito, para a Lição 1 abaixo.

In [ ]:
JANELA = 4096  # ~0,2 s a 20 kHz

def features_da_janela(j):
    x, y, z = j["x"].values, j["y"].values, j["z"].values
    mag = np.sqrt(x**2 + y**2 + z**2)
    mag_ac = mag - mag.mean()
    std_mag = mag_ac.std()
    return {
        "mean_x": x.mean(), "mean_y": y.mean(), "mean_z": z.mean(),
        "std_x":  x.std(),  "std_y":  y.std(),  "std_z":  z.std(),
        "std_mag":   std_mag,
        "p2p_mag":   mag.max() - mag.min(),
        "crest_mag": (np.max(np.abs(mag_ac)) / std_mag) if std_mag > 0 else 0.0,
        "kurt_mag":  pd.Series(mag_ac).kurtosis(),
    }

linhas = []
for label, rpm, caminho in arquivos:
    df = carregar_wfm(caminho)
    n_janelas = len(df) // JANELA
    for k in range(n_janelas):
        j = df.iloc[k*JANELA:(k+1)*JANELA]
        feat = features_da_janela(j)
        feat["label"] = label
        feat["rpm"] = rpm
        linhas.append(feat)

feat_df = pd.DataFrame(linhas)
print(feat_df.groupby(["label", "rpm"]).size())

## 4) Três lições (com motor real, não simulado)

**Lição 1 — `mean_*` é constante em todas as falhas.** É o bias DC do sensor, não o fenômeno.
Confirma empiricamente, com motor de bancada real, a lição do notebook `2.5`: `mean_ax/ay/az`
não carregam informação de vibração nenhuma.

In [ ]:
print(feat_df.groupby("label")[["mean_x", "mean_y", "mean_z"]].mean().round(4))

**Lição 2 — `kurt_mag` resolve falha de rolamento onde `std_mag` mal se move.** Falha de
pista interna é impulsiva (batidas curtas e fortes): a curtose dispara, o desvio padrão sobe
bem menos.

**Lição 3 — eixo empenado é quase invisível no domínio do tempo.** `std_mag`/`kurt_mag` do
eixo empenado ficam próximos do saudável — é o caso que exige olhar o domínio da frequência
(próxima aula: FFT, energia por banda).

In [ ]:
resumo = feat_df.groupby("label")[["std_mag", "kurt_mag"]].mean().round(4)
resumo = resumo.reindex(["saudavel", "eixo_empenado", "desbalanceamento", "rolamento_interno"])
print(resumo)

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
feat_df.boxplot(column="std_mag", by="label", ax=axes[0]); axes[0].set_title("std_mag por classe")
feat_df.boxplot(column="kurt_mag", by="label", ax=axes[1]); axes[1].set_title("kurt_mag por classe")
plt.suptitle(""); plt.tight_layout(); plt.show()

## 5) O contraste de taxa de amostragem: 20 kHz × decimado para 100 Hz

Esta é a pergunta que motivou este notebook: **dá para ver as mesmas falhas amostrando a
100 Hz** (a taxa usada no resto do `app17-IMU`), ou a informação se perde? Decimamos o sinal
completo de cada arquivo (`scipy.signal.decimate`, que já aplica filtro anti-aliasing) de
20 kHz para 100 Hz — fator 200 — e comparamos as MESMAS features, calculadas sobre o sinal
decimado.

Cada arquivo dura 2 s; a 100 Hz isso é só 200 amostras — pouco para uma janela extra, então
usamos **o arquivo inteiro decimado como 1 única janela**. Por isso a comparação de acurácia
abaixo é pequena amostra (5 janelas/classe) e vem rotulada como indicativa, não definitiva — o
que importa comparar são os **valores de feature**, não um número de acurácia isolado.

> **Isto é medição, não a confirmação de uma expectativa.** A hipótese razoável é que o
> desbalanceamento (fenômeno de baixa frequência, 1× RPM) sobrevive à decimação e a falha de
> rolamento (que precisa de banda alta) não — mas o que conta é o que os números abaixo
> mostrarem, não o que a teoria sugere.

**Achado ao implementar isto (deixado aqui de propósito):** decimar em cascata (necessário —
`scipy` recomenda fator ≤13 por chamada) deixa um transiente do filtro IIR nas bordas do sinal.
Num arquivo de 2 s isso é uma fração pequena a 20 kHz, mas a 100 Hz sobra só 200 amostras, e
poucas amostras de transiente já bastam para inflar a curtose (chegou a >100 antes da correção,
contra <1 no sinal físico). A função abaixo descarta 5 amostras de cada borda depois de
decimar — o mesmo princípio do "descartar a transição" do notebook `2.4`, aqui aplicado ao
artefato do filtro em vez de à mão do operador.

In [ ]:
from scipy.signal import decimate

FS_ORIGINAL = 20000
FS_ALVO     = 100
FATOR_TOTAL = FS_ORIGINAL // FS_ALVO  # 200
TRIM_BORDA  = 5   # amostras descartadas de cada ponta (transiente do filtro em cascata)

def decimar_para_100hz(sinal):
    # decimate() recomenda fator <=13 por chamada (estabilidade do filtro IIR);
    # 200 = 10 * 10 * 2, entao decimamos em 3 estagios.
    for fator in (10, 10, 2):
        sinal = decimate(sinal, fator)
    return sinal[TRIM_BORDA:-TRIM_BORDA]

linhas_100hz = []
for label, rpm, caminho in arquivos:
    df = carregar_wfm(caminho)
    x100 = decimar_para_100hz(df["x"].values)
    y100 = decimar_para_100hz(df["y"].values)
    z100 = decimar_para_100hz(df["z"].values)
    janela = pd.DataFrame({"x": x100, "y": y100, "z": z100})
    feat = features_da_janela(janela)
    feat["label"] = label
    feat["rpm"] = rpm
    linhas_100hz.append(feat)

feat_100hz_df = pd.DataFrame(linhas_100hz)
print(f"Amostras por janela a 100 Hz: {len(x100)} (era {JANELA} a 20 kHz)")
print()
print("std_mag e kurt_mag por classe, 20 kHz x 100 Hz decimado:")
comparativo = pd.concat([
    feat_df.groupby("label")[["std_mag", "kurt_mag"]].mean().add_suffix("_20khz"),
    feat_100hz_df.groupby("label")[["std_mag", "kurt_mag"]].mean().add_suffix("_100hz"),
], axis=1)
print(comparativo.reindex(["saudavel", "eixo_empenado", "desbalanceamento", "rolamento_interno"]).round(4))

### Corroborando com um classificador (amostra pequena — indicativo)

`LeaveOneGroupOut` por `rpm` (a mesma lógica do `LeaveOneGroupOut` por `rodada` dos notebooks
`2.x`), rodado nos dois conjuntos de features. Com só 5 janelas por classe a 100 Hz, não é um
benchmark estatisticamente robusto — mas a direção do resultado deve concordar com a tabela de
features acima.

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

FEATURES = ["std_x", "std_y", "std_z", "std_mag", "p2p_mag", "crest_mag", "kurt_mag"]

def avalia_logo_com(df_feats, features):
    X = df_feats[features].values
    y = df_feats["label"].values
    grupos = df_feats["rpm"].values
    logo = LeaveOneGroupOut()
    accs = []
    for idx_tr, idx_te in logo.split(X, y, groups=grupos):
        scaler = StandardScaler().fit(X[idx_tr])
        clf = RandomForestClassifier(n_estimators=50, max_depth=8, random_state=42)
        clf.fit(scaler.transform(X[idx_tr]), y[idx_tr])
        y_pred = clf.predict(scaler.transform(X[idx_te]))
        accs.append(accuracy_score(y[idx_te], y_pred))
    return np.mean(accs), np.std(accs)

def avalia_logo(df_feats):
    return avalia_logo_com(df_feats, FEATURES)

acc_20khz, dp_20khz = avalia_logo(feat_df)
acc_100hz, dp_100hz = avalia_logo(feat_100hz_df)
print(f"20 kHz         : acuracia media (leave-one-rpm-out) = {acc_20khz:.3f} +- {dp_20khz:.3f}  ({len(feat_df)} janelas)")
print(f"100 Hz decimado: acuracia media (leave-one-rpm-out) = {acc_100hz:.3f} +- {dp_100hz:.3f}  ({len(feat_100hz_df)} janelas)")

### Por que a acurácia acima é mais baixa do que se esperaria

Rodando isto pela primeira vez a acurácia a 20 kHz saiu baixa (~0,36 para 4 classes — pouco
acima do acaso, 0,25) — vale investigar por quê em vez de esconder o número. A causa: usamos as
5 velocidades como "rodadas" da mesma classe (seção 1), mas **o RPM sozinho já muda a escala de
quase toda feature**, em qualquer classe — motor girando mais rápido vibra mais, com falha ou
sem. `std_mag`, `p2p_mag` e `crest_mag` sobem com o RPM em TODAS as classes; olhar a tabela da
seção 4 confirma. Ao testar numa velocidade nunca vista (`LeaveOneGroupOut` por `rpm`), essas
features de escala confundem "RPM diferente" com "classe diferente".

`kurt_mag` é diferente: por construção (4º momento normalizado pelo desvio padrão ao quadrado),
é **invariante a escala** — não deveria carregar esse confundimento. Testamos:

In [ ]:
acc_kurt_so, dp_kurt_so = avalia_logo_com(feat_df, ["kurt_mag"])
acc_todas,   dp_todas   = avalia_logo_com(feat_df, FEATURES)
print(f"so kurt_mag (invariante a escala): acc={acc_kurt_so:.3f} +-{dp_kurt_so:.3f}")
print(f"as 7 features (com std_*/p2p/crest, todas dependentes de escala): acc={acc_todas:.3f} +-{dp_todas:.3f}")

**Lição 4 (a mais inesperada deste notebook):** numa validação que atravessa velocidades
diferentes, uma única feature invariante a escala generalizou melhor que o conjunto completo —
as features de amplitude (`std_*`, `p2p_mag`, `crest_mag`) carregam informação de RPM junto
com a de falha, e isso atrapalha fora da velocidade de treino. Nos notebooks `2.x` isso não
aparece porque todas as rodadas de uma classe são coletadas na mesma condição de operação
(o motor do `app17-11` mantém a mesma "velocidade" entre rodadas); aqui, ao usar RPM como
substituto de "rodada", o confundimento veio à tona. É um lembrete de que a escolha de como
agrupar dados em "rodadas"/sessões importa tanto quanto a escolha das features.

## 6) Conclusão

- **`mean_*` constante em toda falha** — confirma com motor real a lição do `2.5`: é bias do
  sensor, não vibração.
- **`kurt_mag` é o indicador de falha impulsiva** (rolamento); `std_mag` sozinho não é
  suficiente para diferenciar as classes.
- **A tabela e a acurácia da seção 5 são a resposta medida** — não assumida — para "que taxa de
  amostragem eu preciso": compare os números de `std_mag_100hz`/`kurt_mag_100hz` com os de
  `_20khz` e a queda (ou não) de acurácia de cada classe, e tire sua própria conclusão sobre
  quais falhas o pipeline de 100 Hz do resto do `app17-IMU` consegue ou não enxergar.
- **Achado que não estava previsto (Lição 4):** ao validar entre velocidades diferentes, uma
  única feature invariante a escala (`kurt_mag`) generalizou melhor que o conjunto completo de
  7 — porque `std_*`/`p2p_mag`/`crest_mag` misturam "RPM mudou" com "classe mudou". É um lembrete
  de que a forma de agrupar dados em "rodadas" pode introduzir confundimentos que só aparecem
  quando você olha o número, não a teoria.
- Isto também é o motivo pelo qual o `app17-11` (5 classes do motor, hardware físico) usa
  **500 Hz**, não 100 Hz: a fundamental de rotação de um motor pequeno já fica perto ou acima
  de 50 Hz (Nyquist a 100 Hz).